# Transiciones ideológicas y choques sobre una red de países

Cuaderno para ejecutar el proyecto entregado. Primero descargue el ZIP `Americas_Markov_codigo_datos_resultados.zip`. En Colab, ejecute la celda de carga y seleccione ese ZIP. En Jupyter local, abra este cuaderno desde la carpeta descomprimida del proyecto.

La ejecución entregada usa DPI 1975–2023 y 31 países. La evaluación tiene 604 casos y 60 cambios; los escenarios parten de enero de 2023. Son escenarios condicionados, no predicciones de gobiernos actuales. Lea README y SOURCES antes de interpretar resultados.

In [ ]:
from pathlib import Path
import io, zipfile

candidates = [Path.cwd(), Path.cwd()/"americas_markov", Path("/content/americas_markov")]
root = next((p for p in candidates if (p/"run.py").exists()), None)
if root is None:
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError("Descomprima el ZIP y abra este cuaderno desde americas_markov.") from exc
    uploaded = files.upload()
    selected = next((name for name in uploaded if name.endswith(".zip")), None)
    if selected is None: raise ValueError("Seleccione el ZIP del proyecto.")
    destination = Path("/content").resolve()
    with zipfile.ZipFile(io.BytesIO(uploaded[selected])) as archive:
        for item in archive.infolist():
            target = (destination/item.filename).resolve()
            if destination not in target.parents: raise ValueError("Ruta insegura dentro del ZIP")
        archive.extractall(destination)
    root = destination/"americas_markov"
if not (root/"run.py").exists(): raise FileNotFoundError("Falta run.py")
print(root)


## Instalar las dependencias

`requirements.txt` admite versiones compatibles. `requirements-lock.txt` registra las versiones utilizadas con Python 3.12; exige Python 3.11 o posterior.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(root/"requirements.txt")], check=True)


## Verificar o actualizar las fuentes

Los datos de la ejecución ya están incluidos. Esta celda usa la caché y vuelve a obtener los archivos COW originales si no se incluyeron en el ZIP. Agregue `--refresh` para descargar nuevas versiones; las cifras podrían cambiar. Puede omitir esta celda y ejecutar el análisis en modo offline.

In [ ]:
subprocess.run([sys.executable, str(root/"run.py"), "--download"], check=True)


## Ejecutar los seis modelos y los escenarios

Imputación y escalado se ajustan exclusivamente dentro de cada entrenamiento. La validación avanza por años. El reloj de simulación es anual, sin fechas electorales futuras. Los estados desconocidos no se convierten en centro.

In [ ]:
subprocess.run([sys.executable, str(root/"run.py"), "--offline", "--run", "--simulate", "--runs", "500", "--horizon", "6", "--seed", "2026"], check=True)


## Comparar resultados

Menor Brier/log-loss es mejor. El riesgo de alternancia es `1 - probabilidad del estado inicial`; AUC y precisión promedio evalúan ese riesgo sin escoger un umbral con las etiquetas futuras. La exactitud cercana al 90 % en la ejecución entregada refleja principalmente continuidad: ninguna clase de máxima probabilidad acertó los 60 cambios.

In [ ]:
import pandas as pd
metrics = pd.read_csv(root/"outputs/main/metricas.csv")
display(metrics[metrics["subset"]=="all"][["model","n","brier_multiclass","log_loss_clipped","brier_change","roc_auc_change","average_precision_change","accuracy"]])
print((root/"outputs/main/RESULTADOS.txt").read_text())


In [ ]:
from IPython.display import Image, display
for name in ["comparacion_modelos", "calibracion", "escenarios"]:
    display(Image(filename=str(root/"outputs/main"/(name+".png"))))


## Sensibilidades opcionales

La carpeta entregada contiene el análisis con conexiones terrestres y marítimas y una ejecución con sólo fronteras terrestres. Para volver a estimar la segunda, use la siguiente celda. `--competitive-only` restringe los orígenes a competitividad electoral EIEC >= 6; es otra muestra y debe compararse con su propia referencia.

In [ ]:
subprocess.run([sys.executable, str(root/"run.py"), "--offline", "--run", "--land-only"], check=True)


## Modificar supuestos y guardar

Edite `src/americas_markov/simulation.py`, diccionario `SCENARIOS`: recesión de crecimiento mundial −2 %, o cambios de petróleo +50 % y alimentos +20 %, durante dos pasos. El escenario base mantiene precios constantes y COVID cero. El código no estima un efecto de Ucrania. Cambiar estos supuestos produce escenarios nuevos; no demuestra causalidad.

Para ampliar el estudio: actualizar la fuente política y construir un calendario de oportunidades de cambio conocido en cada fecha. La serie anual tiene pocos cambios y una alta permanencia institucional.

In [ ]:
import shutil
result_archive = shutil.make_archive(str(root.parent/"Americas_Markov_mi_ejecucion"), "zip", root_dir=root.parent, base_dir=root.name)
print(result_archive)
try:
    from google.colab import files
    files.download(result_archive)
except ImportError:
    pass
